# PACT – phân tích offline trên một run đã train (không train lại)

Notebook này nạp checkpoint (`state.pt`) của một run đã chạy xong rồi chạy các phân tích của 4 hướng tuần này, mỗi phân tích một GPU, song song. Kết quả gom vào một file zip nhỏ để tải về.

**Input cần gắn (Add Input):**

| Input | Nội dung |
|---|---|
| **Output của notebook train** | Version đã chạy `pact_lr5e5_s42` (Add Input → Your Work → notebook train → chọn version). Cần thư mục `outputs/pact_lr5e5_s42/` có `state.pt` và `config.yaml` |
| `pact-code` | Code mới nhất (`python tools/prepare_upload.py`). Ai viết script phân tích mới thì tạo lại zip và upload đè |
| `pact-clip` hoặc `pact-kit` | Trọng số CLIP |
| 11 dataset MTIL | Giống notebook train |

**Các phân tích (bật/tắt ở ô cấu hình):**

| Tên | Người | Câu hỏi | Script |
|---|---|---|---|
| `oracle_snapshot` | Thuận | Có snapshot nào giúp dữ liệu chưa học tốt hơn zero-shot không | `analysis/oracle_snapshot.py` (đã có) |
| `alignment_drift` | Thuận | Alignment ảnh–text có trôi qua các snapshot không (cosine đúng lớp, margin, modality gap, CKA so với CLIP gốc, head học được so với text đóng băng); chỉ số nào dự đoán được Transfer bị giảm | `analysis/alignment_drift.py` (khoảng 15–25 phút) |
| `probe_distance` | Đức Anh | Ma trận khoảng cách giữa MLP và 11 adapter, α theo lưới τ, probe Gaussian (như paper) so với token ảnh thật; tự kiểm tra lại với log train | `analysis/probe_distance.py` (vài phút) |
| `reeval_branch` | Phát | Last/Transfer theo nhánh cond / marg lúc test | `analysis/reeval_branch.py` (đã có) |
| `router_persample` | Khánh | Chất lượng router theo từng ảnh (proto, knn, maha, …) | `analysis/router_persample.py` (đã có) |

**Quy ước cho script mới** (để notebook gọi được mà không phải sửa notebook): đặt ở `analysis/<tên>.py`, nhận `--run-dir <thư mục run>` và `--set <override>` giống `pact.run_mtil`, ghi kết quả (`.json`, `.png`) vào chính `--run-dir`. Script chưa có thì notebook bỏ qua và báo.

Chạy: *Save Version → Save & Run All*.

In [ ]:
# ===================== CẤU HÌNH =====================
RUN_NAME = "pact_lr5e5_s42"   # tên thư mục run trong output của notebook train
PRESET = "full"               # phải khớp run (full: 11 dataset)
MAX_SAMPLES = 2000            # ảnh test mỗi dataset cho các phân tích (0 = toàn bộ, chậm hơn nhiều)
ANALYSES = {                  # tên -> bật/tắt
    "oracle_snapshot": True,   # Thuận
    "alignment_drift": True,   # Thuận (chạy khi đã có script)
    "probe_distance": True,    # Đức Anh (chạy khi đã có script)
    "reeval_branch": False,    # Phát: oracle_hybrid đã có trong results.json; bật khi thử cách gộp adapter mới
    "router_persample": True,  # Khánh: dùng router_features.npz có sẵn nếu tìm thấy, không thì trích lại (1 GPU)
}

WORK = "/kaggle/working"
REPO = f"{WORK}/PACT"
DATA_ROOT = f"{WORK}/data"
OUT = f"{WORK}/analysis_out"
PRESETS = {
    "full": dict(order=["Aircraft", "Caltech101", "CIFAR100", "DTD", "EuroSAT", "Flowers", "Food",
                        "MNIST", "OxfordPet", "StanfordCars", "SUN397"], extra=[]),
    "mid":  dict(order=["Aircraft", "Caltech101", "CIFAR100", "DTD", "EuroSAT", "MNIST", "OxfordPet", "StanfordCars"],
                 extra=["train.iterations=2000", "eval.max_samples=2000", "train.log_every=50"]),
}
ORDER = PRESETS[PRESET]["order"]
print("run:", RUN_NAME, "| preset:", PRESET, "| bật:", [k for k, v in ANALYSES.items() if v])

In [ ]:
# ===================== 1. Tìm pact-kit, cài wheel offline, giải nén repo =====================
import os, sys, json, time, shutil, zipfile, subprocess, importlib

def walk_limited(root, max_depth=6, max_entries=300):
    # os.walk giới hạn độ sâu, bỏ qua thư mục ảnh lớn (không quét hàng trăm nghìn file)
    base = root.rstrip("/").count("/")
    for path, dirs, files in os.walk(root):
        if path.count("/") - base >= max_depth or len(dirs) + len(files) > max_entries:
            dirs[:] = []
        yield path, dirs, files


CLIP_NAME = "ViT-B-16-quickgelu-openai.pt"
KIT_WHEELS = ("open_clip_torch-", "ftfy-")   # chỉ cài đúng 2 wheel này, bỏ qua wheel của input khác
KIT_ZIPS = ("pact-kit.zip", "pact-code.zip", "pact-clip.zip")

def scan(roots):
    # CLIP: lấy ở bất kỳ input nào. Code: mọi thư mục PACT/ (cũ trong pact-kit, mới trong pact-code)
    found = dict(clip=None, codes=[], zips=[])
    for root in roots:
        if not os.path.isdir(root): continue
        for path, dirs, files in walk_limited(root):
            if found["clip"] is None and CLIP_NAME in files:
                found["clip"] = os.path.join(path, CLIP_NAME)
            if os.path.isfile(os.path.join(path, "pact", "engine.py")):
                found["codes"].append(path)
            found["zips"] += [os.path.join(path, f) for f in files if f in KIT_ZIPS]
    return found

def build_id(code_dir):
    try: return json.load(open(os.path.join(code_dir, "BUILD_INFO.json"))).get("id", "?")
    except Exception: return "?"

def build_time(code_dir):   # bản code mới nhất thắng; bản cũ không có BUILD_INFO.json
    try: return json.load(open(os.path.join(code_dir, "BUILD_INFO.json")))["built"]
    except Exception: return ""

ROOTS = ["/kaggle/input", f"{WORK}/_kits"]
found = scan(ROOTS)
extracted = False
for zp in found["zips"]:   # Kaggle giữ nguyên file zip: pact-code (nhỏ) luôn giải nén,
    name = os.path.basename(zp)[:-4]   # zip lớn (pact-kit, pact-clip) chỉ khi còn thiếu CLIP/code
    dest = f"{WORK}/_kits/{name}"
    if os.path.isdir(dest) or not (name == "pact-code" or not found["clip"] or not found["codes"]):
        continue
    print("giải nén", zp, "->", dest)
    with zipfile.ZipFile(zp) as z: z.extractall(dest)
    extracted = True
if extracted:
    found = scan(ROOTS)
assert found["clip"], "Không thấy trọng số CLIP: gắn dataset pact-kit (hoặc pact-clip)"
assert found["codes"], "Không thấy code: gắn dataset pact-code (hoặc pact-kit)"
clip_w = found["clip"]
clip_ref = os.path.join(os.path.dirname(clip_w), "reference.pt")
code_src = max(found["codes"], key=build_time)
CODE_ROOT = os.path.dirname(code_src)          # dataset chứa code; wheel lấy ở đây trước

def find_wheels(roots):
    wheels, wheel_dirs = [], []
    for root in roots:
        for path, dirs, files in walk_limited(root):
            if any(os.path.isfile(os.path.join(path, p, "__init__.py")) for p in ("open_clip", "ftfy")):
                wheel_dirs.append(path)        # Kaggle đã giải nén luôn file .whl
            wheels += [os.path.join(path, f) for f in files
                       if f.endswith(".whl") and f.startswith(KIT_WHEELS)]
    return wheels, wheel_dirs
wheels, wheel_dirs = find_wheels([CODE_ROOT])
if not (wheels or wheel_dirs):
    wheels, wheel_dirs = find_wheels(ROOTS)
wheels = list({os.path.basename(w): w for w in wheels}.values())   # mỗi file 1 lần

print("code     :", code_src, "| build:", build_time(code_src) or "(bản cũ)", "| id:", build_id(code_src),
      f"| {len(found['codes'])} bản tìm thấy")
print("wheels   :", [os.path.basename(w) for w in wheels] or wheel_dirs)
print("clip     :", clip_w, "| reference:", os.path.exists(clip_ref))
assert os.path.exists(clip_ref), "thiếu clip/reference.pt cạnh trọng số CLIP"

if os.path.exists(REPO): shutil.rmtree(REPO)
shutil.copytree(code_src, REPO)
os.chdir(REPO); sys.path.insert(0, REPO)

if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "-q",
                    *wheels], check=True)
elif wheel_dirs:   # wheel đã bị giải nén thành thư mục: dùng trực tiếp qua PYTHONPATH
    sys.path[:0] = wheel_dirs
    os.environ["PYTHONPATH"] = os.pathsep.join(wheel_dirs + [os.environ.get("PYTHONPATH", "")])
importlib.invalidate_caches()
missing = []
for mod in ["torch", "torchvision", "open_clip", "ftfy", "wcwidth", "regex", "timm", "huggingface_hub", "scipy", "yaml"]:
    try:
        m = importlib.import_module(mod); print(f"  {mod:16s} {getattr(m, '__version__', 'ok')}")
    except Exception as e:
        missing.append(mod); print(f"  {mod:16s} THIẾU: {e}")
assert not missing, f"Thiếu thư viện {missing}: thêm wheel của chúng vào pact-kit/wheels"

In [ ]:
# ===================== 2. Phần cứng =====================
import torch
n_gpu = torch.cuda.device_count()
print("GPU:", n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
assert n_gpu >= 1, "Cần ít nhất 1 GPU"
NUM_WORKERS = max(2, min(8, (os.cpu_count() or 4) // max(n_gpu, 1)))
print("CPU:", os.cpu_count(), "| num_workers mỗi phân tích:", NUM_WORKERS,
      "| disk trống (GB):", round(shutil.disk_usage(WORK).free / 2**30, 1))

In [ ]:
# ===================== 3. Kiểm tra trọng số CLIP (đúng QuickGELU, file không hỏng) =====================
r = subprocess.run([sys.executable, "-W", "ignore", "tools/clip_offline.py", "verify",
                    "--weights", clip_w, "--reference", clip_ref], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
assert r.returncode == 0, "Trọng số CLIP không khớp reference"

In [ ]:
# ===================== 4. Dò dataset trong /kaggle/input, dựng data_root (symlink) =====================
# Nếu một dataset không được nhận ra, thêm: "--map", "Tên=/kaggle/input/<thư mục>" để chỉ chỗ tìm.
r = subprocess.run([sys.executable, "tools/kaggle_data.py", "--input", "/kaggle/input",
                    "--data-root", DATA_ROOT, "--datasets", *ORDER, "--check"],
                   capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])
assert r.returncode == 0, "Có dataset thiếu/lỗi (xem ở trên)"

In [ ]:
# ===================== 5. Tham số chung cho mọi run =====================
COMMON = [f"data_root={DATA_ROOT}", "download=false", f"num_workers={NUM_WORKERS}",
          "model.arch=ViT-B-16-quickgelu", f"model.pretrained={clip_w}",
          "mtil.order=[" + ",".join(ORDER) + "]", *PRESETS[PRESET]["extra"]]
print("\n".join(COMMON))

In [ ]:
# ===================== 6. Tìm run đã train trong /kaggle/input =====================
# Input chỉ đọc, nên dựng thư mục run trong /kaggle/working: state.pt là symlink (không chép 1.6 GB),
# các file nhỏ được chép. Kết quả phân tích ghi vào đây.
hits = [p for p, d, f in walk_limited("/kaggle/input", max_depth=8)
        if os.path.basename(p) == RUN_NAME and "state.pt" in f and "config.yaml" in f]
assert hits, (f"Không thấy thư mục {RUN_NAME}/ có state.pt và config.yaml trong /kaggle/input: "
              "gắn output của notebook train (version đã chạy xong)")
SRC_RUN = hits[0]
RUN_DIR = os.path.join(OUT, RUN_NAME); os.makedirs(RUN_DIR, exist_ok=True)
for f in os.listdir(SRC_RUN):
    src, dst = os.path.join(SRC_RUN, f), os.path.join(RUN_DIR, f)
    if os.path.exists(dst) or os.path.isdir(src): continue
    if f.endswith(".pt"): os.symlink(src, dst)
    else: shutil.copy(src, dst)
res = json.load(open(os.path.join(RUN_DIR, "results.json")))
assert res["order"] == ORDER, f"Thứ tự task của run {res['order']} khác PRESET {PRESET}"
print("run:", SRC_RUN, "->", RUN_DIR)
for p in ("oracle_hybrid", "oracle_zeroshot", "oracle_latest", "router"):
    if p in res["protocols"]:
        s = res["protocols"][p]["summary"]
        print(f"  {p:16s} Transfer={s['Transfer']:6.2f}  Avg={s['Avg']:6.2f}  Last={s['Last']:6.2f}")
FEATS = next((os.path.join(p, "router_features.npz") for p, d, f in walk_limited("/kaggle/input", max_depth=8)
              if "router_features.npz" in f), None)
print("router_features.npz:", FEATS or "không có (sẽ trích lại nếu bật router_persample)")

In [ ]:
# ===================== 7. Chạy các phân tích song song (mỗi phân tích 1 GPU) =====================
PY = sys.executable
def gpu_job(script):
    return [PY, script, "--run-dir", RUN_DIR, "--max-samples", str(MAX_SAMPLES), "--set", *COMMON]

JOBS = {}   # tên -> (lệnh, cần GPU)
for name, on in ANALYSES.items():
    if not on: continue
    if name == "router_persample":
        feat = FEATS or os.path.join(RUN_DIR, "router_features.npz")
        steps = [] if FEATS else [[PY, "analysis/router_persample.py", "extract", "--config", "configs/mtil_order1.yaml",
                                   "--out", feat, "--set", *COMMON, f"eval.max_samples={MAX_SAMPLES}"]]
        steps.append([PY, "analysis/router_persample.py", "analyze", feat,
                      "--out", os.path.join(RUN_DIR, "router_persample.json")])
        JOBS[name] = (steps, not FEATS)
        continue
    script = f"analysis/{name}.py"
    if not os.path.exists(os.path.join(REPO, script)):
        print(f"[{name}] chưa có {script} trong pact-code: bỏ qua"); continue
    JOBS[name] = ([gpu_job(script)], True)

procs, gpu_i = {}, 0
for name, (steps, needs_gpu) in JOBS.items():
    env = dict(os.environ)
    if needs_gpu:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_i % n_gpu); gpu_i += 1
    else:
        env["CUDA_VISIBLE_DEVICES"] = ""
    log_f = open(os.path.join(RUN_DIR, f"{name}.log"), "w")
    shell = " && ".join(subprocess.list2cmdline(s) for s in steps)
    procs[name] = (subprocess.Popen(shell, shell=True, cwd=REPO, stdout=log_f, stderr=subprocess.STDOUT, env=env), log_f)
    print(f"[{name}] {'GPU ' + env['CUDA_VISIBLE_DEVICES'] if needs_gpu else 'CPU'} | log: {RUN_DIR}/{name}.log")

T0 = time.time()
while any(p.poll() is None for p, _ in procs.values()):
    time.sleep(60)
    print(time.strftime("%H:%M:%S"), {n: ("chạy" if p.poll() is None else f"xong ({p.returncode})")
                                       for n, (p, _) in procs.items()}, f"{(time.time() - T0) / 60:.0f} phút")
for name, (p, log_f) in procs.items():
    log_f.close()
    print(f"\n===== {name} (exit {p.returncode})")
    print("".join(open(os.path.join(RUN_DIR, f"{name}.log")).readlines()[-30:]))

In [ ]:
# ===================== 8. Đóng gói kết quả phân tích =====================
# Gồm mọi file trong thư mục run trừ checkpoint (.pt) và file > 50 MB. Tải: tab Output -> file .zip.
bundle = f"{WORK}/pact_analysis_{RUN_NAME}_{time.strftime('%Y%m%d_%H%M')}.zip"
skipped = []
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(RUN_DIR)):
        full = os.path.join(RUN_DIR, f)
        if os.path.isdir(full) or f.endswith(".pt") or os.path.getsize(full) > 50 * 2**20:
            skipped.append(f); continue
        z.write(full, os.path.join(RUN_NAME, f))
    z.writestr("analysis_info.json", json.dumps({
        "created": time.strftime("%Y-%m-%d %H:%M:%S"), "run": RUN_NAME, "source": SRC_RUN,
        "analyses": {n: procs[n][0].returncode for n in procs}, "max_samples": MAX_SAMPLES,
        "code_build": (lambda f: json.load(open(f)) if os.path.exists(f) else None)(f"{REPO}/BUILD_INFO.json"),
    }, indent=2, ensure_ascii=False))
with zipfile.ZipFile(bundle) as z: print(bundle, f"{os.path.getsize(bundle) / 2**20:.1f} MB:", z.namelist())
print("bỏ qua:", skipped)
from IPython.display import FileLink, display
cwd = os.getcwd(); os.chdir(WORK); display(FileLink(os.path.basename(bundle))); os.chdir(cwd)